In [ ]:
%sql
-- ============================================================
-- Restockify Workflow — §4.2 Deep Analysis, as Unity Catalog functions
-- ============================================================
-- Registers the Genie Agent's deep-analysis logic (consumption trend,
-- stockout forecast, urgency classification, quote line-item math, veto)
-- as governed Unity Catalog SQL functions. The functions themselves are
-- registered in ab_training.agentic_restock (the schema we own), but their
-- bodies read Data Engineering's real gold_dev star schema:
--   - gold_dev.dim.dim_part / dim_warehouse / dim_plant — business-key lookups
--   - gold_dev.supply_chain_analytics.fact_inventory_snapshot — current stock,
--     SAFETY_STOCK_QTY (reorder trigger), MAX_STOCK_LEVEL (restock target),
--     STOCKOUT_RISK. Daily snapshot grain, so callers always take the MOST
--     RECENT SNAPSHOT_DATE_KEY per part/warehouse (via MAX_BY), never a raw MAX().
--   - gold_dev.supply_chain_analytics.fact_inventory_transaction — ISSUE-type
--     rows are the consumption events (replaces the old mock consumption_history).
--   - gold_dev.supply_chain_analytics.fact_procurement — open purchase orders,
--     used by the (now real, not stubbed) needs_restock veto and the new
--     avg_lead_time_days function.
--
-- All six functions keep their original item_id/warehouse_id-shaped external
-- signature (now `part_id`/`warehouse_id`, since the gold_dev business keys
-- are named PART_ID/WAREHOUSE_ID) so Genie and the Supervisor Agent don't
-- need to reason about surrogate keys (PART_KEY/WAREHOUSE_KEY) at all.
--
-- Why UC functions instead of a notebook/job task: per Databricks Agent
-- Bricks docs, this is the correct primitive for "complex logic that
-- cannot be captured with a static or parameterized SQL query" — they can
-- be registered as trusted SQL functions on a Genie Agent, added directly
-- as tools on a Supervisor Agent, and queried by anyone with EXECUTE
-- permission, all without duplicating the logic in Python.
-- ============================================================

In [ ]:
%sql
-- ============================================================
-- Function 1: avg_daily_consumption
-- Trailing-window average daily consumption, anchored to today
-- (current_date()). Reads gold_dev.supply_chain_analytics.
-- fact_inventory_transaction directly (TRANSACTION_TYPE = 'ISSUE' rows
-- are consumption events) rather than fact_inventory_snapshot's
-- precomputed AVG_DAILY_CONSUMPTION column, so the `lookback_days`
-- parameter stays meaningful and auditable instead of trusting an
-- opaque, differently-windowed DE aggregate.
-- NOTE for this dev dataset: transaction rows are seeded for a fixed
-- 2026-08-11..2026-08-20 range, so as real time moves past that range
-- the window naturally covers fewer seed days — expected behavior for
-- a static dev dataset, not a bug. Kept as a single flat query (no
-- subqueries/window functions) because Databricks SQL functions reject
-- correlated subqueries once another function calls this one and its
-- body gets inlined.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.avg_daily_consumption(
  part_id STRING COMMENT 'Part business key, e.g. P1001 (gold_dev.dim.dim_part.PART_ID)',
  warehouse_id STRING COMMENT 'Warehouse business key, e.g. WH001 (gold_dev.dim.dim_warehouse.WAREHOUSE_ID)',
  lookback_days INT DEFAULT 14 COMMENT 'Trailing window size in days, ending today'
)
RETURNS DOUBLE
COMMENT 'Average daily consumption (architecture §4.2) over the trailing `lookback_days` ending today, computed from ISSUE-type rows in gold_dev.supply_chain_analytics.fact_inventory_transaction. Returns 0.0 if no consumption transactions exist in that window.'
RETURN
  SELECT COALESCE(SUM(fit.QUANTITY), 0.0) / avg_daily_consumption.lookback_days
  FROM gold_dev.supply_chain_analytics.fact_inventory_transaction fit
  JOIN gold_dev.dim.dim_part dp ON fit.PART_KEY = dp.PART_KEY
  JOIN gold_dev.dim.dim_warehouse dw ON fit.WAREHOUSE_KEY = dw.WAREHOUSE_KEY
  WHERE dp.PART_ID = avg_daily_consumption.part_id
    AND dw.WAREHOUSE_ID = avg_daily_consumption.warehouse_id
    AND fit.TRANSACTION_TYPE = 'ISSUE'
    AND to_date(CAST(fit.TRANSACTION_DATE_KEY AS STRING), 'yyyyMMdd') > date_sub(current_date(), avg_daily_consumption.lookback_days);

In [ ]:
%sql
-- ============================================================
-- Function 2: predicted_stockout_date
-- Forward-looking forecast from *today* (real time), using the most
-- recent snapshot's on-hand stock (fact_inventory_snapshot is a daily
-- snapshot fact, not a single current-state row, so MAX_BY picks the
-- latest SNAPSHOT_DATE_KEY per part/warehouse) and the trailing
-- avg_daily_consumption.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.predicted_stockout_date(
  part_id STRING COMMENT 'Part business key',
  warehouse_id STRING COMMENT 'Warehouse business key'
)
RETURNS DATE
COMMENT 'Earliest predicted stockout date (architecture §4.2), projected from today using the latest fact_inventory_snapshot QUANTITY_ON_HAND and the trailing 14-day avg_daily_consumption. NULL when consumption is ~0 (nothing to forecast).'
RETURN
  -- MAX(...) wrappers around nested function calls are required:
  -- Databricks SQL functions reject a table-scanning body whose SELECT
  -- list isn't provably single-row via aggregation once the function is
  -- called from inside another function's body — a PK-filtered WHERE
  -- alone isn't accepted as proof. MAX_BY(..., SNAPSHOT_DATE_KEY) is
  -- itself an aggregate, so it satisfies the same requirement while also
  -- picking the latest snapshot row.
  SELECT
    CASE
      WHEN MAX(ab_training.agentic_restock.avg_daily_consumption(predicted_stockout_date.part_id, predicted_stockout_date.warehouse_id, 14)) > 0
      THEN date_add(
        current_date(),
        CAST(CEIL(
          MAX_BY(fis.QUANTITY_ON_HAND, fis.SNAPSHOT_DATE_KEY)
          / MAX(ab_training.agentic_restock.avg_daily_consumption(predicted_stockout_date.part_id, predicted_stockout_date.warehouse_id, 14))
        ) AS INT)
      )
      ELSE NULL
    END
  FROM gold_dev.supply_chain_analytics.fact_inventory_snapshot fis
  JOIN gold_dev.dim.dim_part dp ON fis.PART_KEY = dp.PART_KEY
  JOIN gold_dev.dim.dim_warehouse dw ON fis.WAREHOUSE_KEY = dw.WAREHOUSE_KEY
  WHERE dp.PART_ID = predicted_stockout_date.part_id
    AND dw.WAREHOUSE_ID = predicted_stockout_date.warehouse_id;

In [ ]:
%sql
-- ============================================================
-- Function 3: classify_urgency
-- Pure classification, no table access. The gold_dev star schema has
-- no minimum_stock_qty-style absolute floor column, so the "always
-- CRITICAL regardless of forecast" override now uses Data Engineering's
-- own precomputed fact_inventory_snapshot.STOCKOUT_RISK = 'HIGH' signal
-- instead; otherwise urgency is banded by days_remaining until stockout,
-- same as before.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.classify_urgency(
  stockout_risk STRING COMMENT 'Latest fact_inventory_snapshot.STOCKOUT_RISK for this part/warehouse (LOW/MEDIUM/HIGH)',
  days_remaining DOUBLE COMMENT 'Days until predicted stockout, or NULL if no forecast (near-zero consumption)'
)
RETURNS STRING
COMMENT 'Urgency classification per architecture §4.2: CRITICAL (fact_inventory_snapshot.STOCKOUT_RISK = HIGH, or <=3 days to stockout), HIGH (<=7 days), MEDIUM (<=14 days), LOW (>14 days or no forecastable consumption).'
RETURN
  CASE
    WHEN classify_urgency.stockout_risk = 'HIGH' THEN 'CRITICAL'
    WHEN classify_urgency.days_remaining IS NULL THEN 'LOW'
    WHEN classify_urgency.days_remaining <= 3 THEN 'CRITICAL'
    WHEN classify_urgency.days_remaining <= 7 THEN 'HIGH'
    WHEN classify_urgency.days_remaining <= 14 THEN 'MEDIUM'
    ELSE 'LOW'
  END;

In [ ]:
%sql
-- ============================================================
-- Function 4: requested_restock_qty
-- Quote line-item math: how many units to order to reach the restock
-- target. Floored at 0 (never a negative order). MAX_STOCK_LEVEL takes
-- over the role the old mock threshold_config_table.target_stock_qty
-- played; both come from the latest fact_inventory_snapshot row.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.requested_restock_qty(
  part_id STRING COMMENT 'Part business key',
  warehouse_id STRING COMMENT 'Warehouse business key'
)
RETURNS INT
COMMENT 'Suggested restock quantity: MAX_STOCK_LEVEL - QUANTITY_ON_HAND (from the latest fact_inventory_snapshot row), floored at 0. NULL if the part/warehouse has no snapshot rows.'
RETURN
  SELECT MAX(GREATEST(
    MAX_BY(fis.MAX_STOCK_LEVEL, fis.SNAPSHOT_DATE_KEY) - MAX_BY(fis.QUANTITY_ON_HAND, fis.SNAPSHOT_DATE_KEY),
    0
  ))
  FROM gold_dev.supply_chain_analytics.fact_inventory_snapshot fis
  JOIN gold_dev.dim.dim_part dp ON fis.PART_KEY = dp.PART_KEY
  JOIN gold_dev.dim.dim_warehouse dw ON fis.WAREHOUSE_KEY = dw.WAREHOUSE_KEY
  WHERE dp.PART_ID = requested_restock_qty.part_id
    AND dw.WAREHOUSE_ID = requested_restock_qty.warehouse_id;

In [ ]:
%sql
-- ============================================================
-- Function 5: needs_restock (veto power)
-- Genie Agent's veto: is this Lakeflow-flagged candidate a false
-- positive? No longer a stub — gold_dev.supply_chain_analytics.
-- fact_procurement gives us real open-purchase-order data. Vetoes
-- (returns FALSE) when an open PO (STATUS IN ISSUED/PARTIAL) at the
-- warehouse's linked plant already has enough PENDING_QTY to cover the
-- suggested reorder quantity; otherwise TRUE (genuinely needs restocking).
-- Falls back to TRUE (never veto) when the warehouse has no linked plant
-- (e.g. REGIONAL SPARES warehouses per dim_warehouse.LINKED_PLANT_ID) or
-- no open PO exists — we can't confirm coverage, so we don't suppress.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.needs_restock(
  part_id STRING COMMENT 'Part business key',
  warehouse_id STRING COMMENT 'Warehouse business key'
)
RETURNS BOOLEAN
COMMENT 'Veto decision (architecture §4.2): FALSE when an open (ISSUED/PARTIAL) purchase order for this part at the warehouse''s linked plant already has enough PENDING_QTY to cover requested_restock_qty (a false positive -- restocking is already in flight); TRUE otherwise (genuinely needs restocking, or coverage cannot be confirmed).'
RETURN
  SELECT COALESCE(
    MAX(ab_training.agentic_restock.requested_restock_qty(needs_restock.part_id, needs_restock.warehouse_id))
      > SUM(COALESCE(fp.PENDING_QTY, 0)),
    TRUE
  )
  FROM gold_dev.dim.dim_warehouse dw
  LEFT JOIN gold_dev.dim.dim_plant dpl
    ON dpl.PLANT_ID = dw.LINKED_PLANT_ID
  LEFT JOIN gold_dev.dim.dim_part dp
    ON dp.PART_ID = needs_restock.part_id
  LEFT JOIN gold_dev.supply_chain_analytics.fact_procurement fp
    ON fp.PLANT_KEY = dpl.PLANT_KEY
    AND fp.PART_KEY = dp.PART_KEY
    AND fp.STATUS IN ('ISSUED', 'PARTIAL')
  WHERE dw.WAREHOUSE_ID = needs_restock.warehouse_id;

In [ ]:
%sql
-- ============================================================
-- Function 6: restock_candidate_summary
-- Deterministic natural-language one-liner combining functions
-- 1-4. This is the function most useful as a Genie Agent trusted
-- asset (answers "why does X need restocking?") and as the text
-- source for the Teams Adaptive Card / quote_metadata.summary_report.
-- NOTE: gold_dev has no unit_of_measure column on dim_part or
-- fact_inventory_snapshot (unlike the old mock inventory_stock_level),
-- so quantities are reported as plain "units" here.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.restock_candidate_summary(
  part_id STRING COMMENT 'Part business key',
  warehouse_id STRING COMMENT 'Warehouse business key'
)
RETURNS STRING
COMMENT 'Deterministic natural-language summary of one restock candidate: stock on hand, avg daily consumption, predicted stockout date, urgency, and suggested reorder quantity (architecture §4.2).'
RETURN
  SELECT MAX(CONCAT(
    dp.PART_NAME, ' at ', dw.WAREHOUSE_CODE, ': ',
    CAST(MAX_BY(fis.QUANTITY_ON_HAND, fis.SNAPSHOT_DATE_KEY) AS STRING), ' units on hand (safety stock ',
    CAST(MAX_BY(fis.SAFETY_STOCK_QTY, fis.SNAPSHOT_DATE_KEY) AS STRING), '). Avg consumption ',
    CAST(ROUND(ab_training.agentic_restock.avg_daily_consumption(restock_candidate_summary.part_id, restock_candidate_summary.warehouse_id, 14), 1) AS STRING),
    '/day. ',
    CASE
      WHEN ab_training.agentic_restock.predicted_stockout_date(restock_candidate_summary.part_id, restock_candidate_summary.warehouse_id) IS NOT NULL
      THEN CONCAT('Predicted stockout ', CAST(ab_training.agentic_restock.predicted_stockout_date(restock_candidate_summary.part_id, restock_candidate_summary.warehouse_id) AS STRING), '. ')
      ELSE 'No forecastable stockout (near-zero consumption). '
    END,
    'Urgency: ', ab_training.agentic_restock.classify_urgency(
      MAX_BY(fis.STOCKOUT_RISK, fis.SNAPSHOT_DATE_KEY),
      datediff(ab_training.agentic_restock.predicted_stockout_date(restock_candidate_summary.part_id, restock_candidate_summary.warehouse_id), current_date())
    ), '. ',
    'Suggested reorder: ', CAST(ab_training.agentic_restock.requested_restock_qty(restock_candidate_summary.part_id, restock_candidate_summary.warehouse_id) AS STRING), ' units.'
  ))
  FROM gold_dev.supply_chain_analytics.fact_inventory_snapshot fis
  JOIN gold_dev.dim.dim_part dp ON fis.PART_KEY = dp.PART_KEY
  JOIN gold_dev.dim.dim_warehouse dw ON fis.WAREHOUSE_KEY = dw.WAREHOUSE_KEY
  WHERE dp.PART_ID = restock_candidate_summary.part_id
    AND dw.WAREHOUSE_ID = restock_candidate_summary.warehouse_id;

In [ ]:
%sql
-- ============================================================
-- Verification: run all 6 functions against every candidate the
-- §4.1 coarse check would flag (QUANTITY_ON_HAND <= SAFETY_STOCK_QTY on
-- the latest fact_inventory_snapshot row per part/warehouse). This
-- mirrors src/agentic_restock/jobs/lakeflow_trigger.py's
-- build_coarse_check_query() -- eyeball these against the real gold_dev
-- data seeded by Data Engineering.
-- ============================================================

WITH latest_snapshot AS (
  SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY PART_KEY, WAREHOUSE_KEY ORDER BY SNAPSHOT_DATE_KEY DESC) AS rn
  FROM gold_dev.supply_chain_analytics.fact_inventory_snapshot
)
SELECT
  dp.PART_ID AS part_id,
  dp.PART_NAME AS part_name,
  dw.WAREHOUSE_ID AS warehouse_id,
  ls.QUANTITY_ON_HAND AS current_stock_qty,
  ls.SAFETY_STOCK_QTY AS reorder_point_qty,
  ls.STOCKOUT_RISK AS stockout_risk,
  ROUND(ab_training.agentic_restock.avg_daily_consumption(dp.PART_ID, dw.WAREHOUSE_ID, 14), 2) AS avg_daily_consumption,
  ab_training.agentic_restock.predicted_stockout_date(dp.PART_ID, dw.WAREHOUSE_ID) AS predicted_stockout_date,
  ab_training.agentic_restock.classify_urgency(
    ls.STOCKOUT_RISK,
    datediff(ab_training.agentic_restock.predicted_stockout_date(dp.PART_ID, dw.WAREHOUSE_ID), current_date())
  ) AS urgency_level,
  ab_training.agentic_restock.requested_restock_qty(dp.PART_ID, dw.WAREHOUSE_ID) AS requested_restock_qty,
  ab_training.agentic_restock.needs_restock(dp.PART_ID, dw.WAREHOUSE_ID) AS needs_restock,
  ab_training.agentic_restock.restock_candidate_summary(dp.PART_ID, dw.WAREHOUSE_ID) AS summary
FROM latest_snapshot ls
JOIN gold_dev.dim.dim_part dp ON ls.PART_KEY = dp.PART_KEY AND dp.IS_CURRENT = true
JOIN gold_dev.dim.dim_warehouse dw ON ls.WAREHOUSE_KEY = dw.WAREHOUSE_KEY
WHERE ls.rn = 1
  AND dp.LIFECYCLE_STATUS = 'ACTIVE'
  AND dw.OPERATIONAL_STATUS = 'ACTIVE'
  AND ls.QUANTITY_ON_HAND <= ls.SAFETY_STOCK_QTY
ORDER BY urgency_level, part_id;

In [ ]:
%sql
-- ============================================================
-- Function 7 (new): avg_lead_time_days
-- The old mock threshold_config_table.lead_time_days config field has no
-- equivalent anywhere in the gold_dev star schema -- dim_part, dim_supplier,
-- and fact_procurement were all checked and none carry a fixed lead-time
-- config. This function derives an *empirical* estimate instead, from
-- historical purchase orders: EXPECTED_DATE_KEY - ORDER_DATE_KEY, averaged
-- across all of a part's POs (any supplier/plant). Not a contracted SLA --
-- purely informational, surfaced in Teams/Genie text, not used by
-- classify_urgency or the veto. Returns NULL if the part has no
-- procurement history.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.avg_lead_time_days(
  part_id STRING COMMENT 'Part business key, e.g. P1001'
)
RETURNS DOUBLE
COMMENT 'Empirical average supplier lead time in days for a part, derived from gold_dev.supply_chain_analytics.fact_procurement (EXPECTED_DATE_KEY - ORDER_DATE_KEY, averaged across all historical POs for the part). Not a contracted SLA -- the gold_dev star schema has no fixed lead_time_days config field. NULL if the part has no procurement history.'
RETURN
  SELECT AVG(
    datediff(
      to_date(CAST(fp.EXPECTED_DATE_KEY AS STRING), 'yyyyMMdd'),
      to_date(CAST(fp.ORDER_DATE_KEY AS STRING), 'yyyyMMdd')
    )
  )
  FROM gold_dev.supply_chain_analytics.fact_procurement fp
  JOIN gold_dev.dim.dim_part dp ON fp.PART_KEY = dp.PART_KEY
  WHERE dp.PART_ID = avg_lead_time_days.part_id;

-- Standalone verification: average lead time for the first 5 parts with
-- procurement history.
SELECT DISTINCT
  dp.PART_ID,
  dp.PART_NAME,
  ROUND(ab_training.agentic_restock.avg_lead_time_days(dp.PART_ID), 1) AS avg_lead_time_days
FROM gold_dev.supply_chain_analytics.fact_procurement fp
JOIN gold_dev.dim.dim_part dp ON fp.PART_KEY = dp.PART_KEY
ORDER BY dp.PART_ID
LIMIT 5;